# Mock lab demo: one weld at a time

This notebook stands in for the lab. Each button is one physical step:

1. **Weld**: makes the weld (saved as `_mock/welds/<weld_id>.json`).
2. **Visual inspection**: gives the 0 to 3 rating. A 0 means code D1: stop.
3. **Read stress P1/P2/P3**: one reading per click. Click again to re-read a rough spot (up to 3 fit the sheet).
4. **Cut specimens**: cuts the tensile specimens; this destroys the weld for stress, so read every position first.
5. **Tensile specimen 1/2**: destructive, so each specimen can be tested once.

The widget does **not** write to the spreadsheet. Type each value from the right-hand panel into
`A1/data/experiments.xlsx` (yellow columns), save, close Excel, then run the check cell at the bottom.


In [9]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "_mock").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from _mock import fsw
from _mock.lab import MockLab
from _mock.lab_widget import LabDemo

EXPERIMENT_FILE = ROOT / "A1" / "data" / "experiments.xlsx"
CAMPAIGN_SEED = (
    0  # same as A1/A01_mock_fill.py, so demo and dry run give the same numbers
)
MOCK_PARAMS = fsw.MockParams(drift_stress_per_weld=0.0)

lab = MockLab(
    ROOT / "_mock" / "welds", campaign_seed=CAMPAIGN_SEED, params=MOCK_PARAMS
)

In [10]:
LabDemo(EXPERIMENT_FILE, lab).show()

## Check the spreadsheet

Save and close Excel first. This runs `A1/A01.py`, which validates the entries, fills the grey
computed columns and shows what is still missing.


In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, str(ROOT / "A1" / "A01.py")], capture_output=True, text=True
)
print(result.stdout, result.stderr)

## Reset the demo

Set `RESET = True` and run to start over: deletes every mock weld and moves the current
spreadsheet into `A1/data/backups/`. Then run `python A1/A01.py` to create a fresh design.
**Do not use this once the real campaign has started.**


In [12]:
RESET = False

if RESET:
    import shutil
    from datetime import datetime

    for f in lab.list():
        lab.discard(f)
    if EXPERIMENT_FILE.exists():
        target = (
            EXPERIMENT_FILE.parent
            / "backups"
            / f"experiments_before_reset_{datetime.now():%Y%m%d-%H%M%S}.xlsx"
        )
        target.parent.mkdir(exist_ok=True)
        shutil.move(EXPERIMENT_FILE, target)
        print(f"Spreadsheet moved to {target}")
    print(
        "Mock welds cleared. Run the check cell (or python A1/A01.py) to create a fresh design."
    )